In [ ]:
from google.colab import files
files.upload()

In [ ]:
import json

def convert_spacy_to_tf_format(spacy_data):
    converted = []
    for entry in spacy_data:
        if isinstance(entry, list):
            text, ann = entry
            label_seq = ['O'] * len(text)
            for start, end, label in ann['entities']:
                for i in range(start, end):
                    if label == 'MONTH':
                        label_seq[i] = 'M'
                    elif label == 'DATE':
                        label_seq[i] = 'D'
                    elif label == 'YEAR':
                        label_seq[i] = 'Y'
            converted.append({
                "text": list(text),
                "labels": label_seq
            })
        elif isinstance(entry, dict):
            converted.append(entry)
    return converted

with open("merged_train.json", "r", encoding="utf-8") as f:
    mixed_data = json.load(f)

cleaned_data = convert_spacy_to_tf_format(mixed_data)

with open("cleaned_merged_train.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

print("Cleaned and saved as 'cleaned_merged_train.json'")


Cleaned and saved as 'cleaned_merged_train.json'


In [ ]:
import json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

with open("/content/cleaned_merged_train.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [item["text"] for item in data]
labels = [item["labels"] for item in data]

all_chars = sorted(set(char for seq in texts for char in seq))
all_labels = sorted(set(lbl for seq in labels for lbl in seq))

char2idx = {c: i + 1 for i, c in enumerate(all_chars)}
label2idx = {l: i for i, l in enumerate(all_labels)}
idx2label = {i: l for l, i in label2idx.items()}

X = [[char2idx[c] for c in seq] for seq in texts]
y = [[label2idx[l] for l in seq] for seq in labels]

max_len = max(len(seq) for seq in X)
X_padded = pad_sequences(X, maxlen=max_len, padding="post")
y_padded = pad_sequences(y, maxlen=max_len, padding="post")
y_categorical = [to_categorical(i, num_classes=len(label2idx)) for i in y_padded]

X_train, X_val, y_train, y_val = train_test_split(X_padded, y_categorical, test_size=0.1, random_state=42)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(char2idx)+1, output_dim=64, input_length=max_len),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(units=64, return_sequences=True)),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(label2idx), activation='softmax'))
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

model.fit(
    X_train, np.array(y_train),
    validation_data=(X_val, np.array(y_val)),
    batch_size=27,
    epochs=10,
    verbose=1
)

model.save("date_ner_bilstm_model.h5")
with open("char2idx.json", "w") as f: json.dump(char2idx, f)
with open("label2idx.json", "w") as f: json.dump(label2idx, f)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.5914 - loss: 0.9679 - val_accuracy: 0.8395 - val_loss: 0.4388
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.8803 - loss: 0.3432 - val_accuracy: 0.9829 - val_loss: 0.1157
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9852 - loss: 0.0868 - val_accuracy: 0.9931 - val_loss: 0.0383
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.9928 - loss: 0.0371 - val_accuracy: 0.9940 - val_loss: 0.0291
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.9918 - loss: 0.0326 - val_accuracy: 0.9949 - val_loss: 0.0227
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.9918 - loss: 0.0275 - val_accuracy: 0.9949 - val_loss: 0.0184
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.9950 - loss: 0.0183 - val_accuracy: 0.9973 - val_loss: 0.0148
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9965 - loss: 0.0167 - val_accuracy: 0.9979 - v

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np
import json

model = load_model("date_ner_bilstm_model.h5")
with open("char2idx.json") as f: char2idx = json.load(f)
with open("label2idx.json") as f: label2idx = json.load(f)
idx2label = {v: k for k, v in label2idx.items()}


In [ ]:
from datetime import datetime

def post_process(predicted_tags):
    day = ''
    month = ''
    year = ''

    for ch, tag in predicted_tags:
        if tag == 'D':
            day += ch
        elif tag == 'M':
            month += ch
        elif tag == 'Y':
            year += ch

    try:
        if not month.isdigit():
            month_num = datetime.strptime(month[:3], "%b").month
        else:
            month_num = int(month)
    except Exception:
        return "Invalid Date"

    try:
        day_num = int(day)
        year_num = int(year)

        final_date = datetime(year=year_num, month=month_num, day=day_num)
        return final_date.strftime("%d-%m-%Y")
    except Exception:
        return "Invalid Date"


In [ ]:
def predict_tags(text):
    input_seq = [char2idx.get(c, 0) for c in text]
    padded = pad_sequences([input_seq], maxlen=model.input_shape[1], padding='post')
    pred = model.predict(padded)[0]
    pred_tags = [idx2label[np.argmax(p)] for p in pred[:len(text)]]
    return list(zip(text, pred_tags))

predicted_tags = predict_tags("10 march 2005")
print(predicted_tags)
print(post_process(predicted_tags))



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
[('1', 'D'), ('0', 'D'), (' ', 'O'), ('m', 'M'), ('a', 'M'), ('r', 'M'), ('c', 'M'), ('h', 'M'), (' ', 'O'), ('2', 'Y'), ('0', 'Y'), ('0', 'Y'), ('5', 'Y')]
10-03-2005


In [ ]:


import tensorflow as tf
import numpy as np
import json
from datetime import datetime

with open("char2idx.json") as f:
    char2idx = json.load(f)
with open("label2idx.json") as f:
    label2idx = json.load(f)
idx2label = {v: k for k, v in label2idx.items()}
model = tf.keras.models.load_model("date_ner_bilstm_model.h5")
max_len = model.input_shape[1]
def post_process(text_tensor, pred_tensor):
    text = text_tensor.numpy().decode("utf-8")
    pred = pred_tensor.numpy()

    tags = [idx2label.get(int(i), "O") for i in pred[:len(text)]]

    day, month, year = "", "", ""
    for ch, tag in zip(text, tags):
        if tag == 'D':
            day += ch
        elif tag == 'M':
            month += ch
        elif tag == 'Y':
            year += ch

    try:
        if not month.isdigit():
            month_num = datetime.strptime(month[:3], "%b").month
        else:
            month_num = int(month)
    except:
        return b"Invalid Date"

    try:
        final_date = datetime(int(year), int(month_num), int(day))
        return final_date.strftime("%d-%m-%Y").encode("utf-8")
    except:
        return b"Invalid Date"
class DateNERValidator(tf.Module):
    def __init__(self):
        super().__init__()
        self.model = model
        self.char2idx = char2idx

    @tf.function(input_signature=[tf.TensorSpec(shape=(), dtype=tf.string)])
    def predict_date(self, input_text):
        chars = tf.strings.unicode_split(input_text, 'UTF-8')[:max_len]
        def lookup(c):
            return self.char2idx.get(c.numpy().decode("utf-8"), 0)

        ids = tf.map_fn(
            lambda c: tf.py_function(lookup, [c], Tout=tf.int32),
            chars,
            dtype=tf.int32
        )

        padded = tf.pad(ids, [[0, max_len - tf.shape(ids)[0]]])
        inputs = tf.expand_dims(padded, 0)
        logits = self.model(inputs, training=False)
        preds = tf.argmax(logits, axis=-1)[0]

        final = tf.py_function(post_process, [input_text, preds], tf.string)
        return final
ner_validator = DateNERValidator()
tf.saved_model.save(ner_validator, "exported_date_validator", signatures={"predict_date": ner_validator.predict_date})

print("Exported full TensorFlow model to 'exported_date_validator'")


Exported full TensorFlow model to 'exported_date_validator'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')